In [1]:
!ls /kaggle/input/notebooks/aadiparkhi
!ls /kaggle/input/datasets/*
!ls /kaggle/input/notebooks/aadiparkhi/*02*

01a2-bdd-curation  01a-data-curation  02-corruption-generation
drift-multitask-checkpoints
base_bdd.csv	 corrupt_bdd_manifest.csv     __notebook__.ipynb
base_places.csv  corrupt_places_manifest.csv  __output__.json
corrupt		 custom.css		      __results__.html


In [2]:
import os, glob, json, numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from transformers import AutoModel

dev, NB = 'cuda', '/kaggle/input/notebooks/aadiparkhi'
CK = glob.glob('/kaggle/input/datasets/*/drift-multitask-checkpoints')[0]
C = {'places': f'{NB}/01a-data-curation/data', 'bdd': f'{NB}/01a2-bdd-curation/data'}
G = f'{NB}/02-corruption-generation'
pl = pd.read_csv(f'{NB}/01a-data-curation/places_manifest.csv')
bd = pd.read_csv(f'{NB}/01a2-bdd-curation/bdd_manifest.csv')
print(CK)

class M(nn.Module):
    def __init__(s, bb, d, dino=False):
        super().__init__()
        s.bb, s.dino = bb, dino
        s.h1, s.h2, s.h3 = nn.Linear(d, 48), nn.Linear(d, 11), nn.Linear(d, 2)
    def forward(s, x):
        if s.dino:
            h = s.bb(pixel_values=x).last_hidden_state
            z = torch.cat([h[:, 0], h[:, 1:].mean(1)], 1)
        else:
            z = s.bb(x)
        return z, s.h1(z), s.h2(z), s.h3(z)

r18 = models.resnet18(weights=None); r18.fc = nn.Identity()
r50 = models.resnet50(weights=None); r50.fc = nn.Identity()
cx = models.convnext_tiny(weights=None); cx.classifier[2] = nn.Identity()
nets = {'resnet18': M(r18, 512), 'resnet50': M(r50, 2048), 'convnext': M(cx, 768),
        'dino': M(AutoModel.from_pretrained('facebook/dinov2-small'), 768, True)}
for k, m in nets.items():
    m.load_state_dict(torch.load(f'{CK}/multitask_{k}.pt', map_location=dev)); m.to(dev).eval()

tf = T.Compose([T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
class PD(Dataset):
    def __init__(s, p): s.p = p
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return tf(Image.open(s.p[i]).convert('RGB'))

@torch.no_grad()
def extract(paths, dom, bs=128):
    out = {k: {'emb': [], 'log': []} for k in nets}
    for x in DataLoader(PD(paths), batch_size=bs, num_workers=4):
        x = x.to(dev)
        for k, m in nets.items():
            with torch.autocast('cuda'): z, a, b, c = m(x)
            out[k]['emb'].append(z.float().cpu().half())
            out[k]['log'].append((a if dom == 'places' else torch.cat([b, c], 1)).float().cpu())
    return {k: {n: torch.cat(v).numpy() for n, v in d.items()} for k, d in out.items()}

s = pl[pl.split == 'val'].sample(480, random_state=0)
r = extract([f"{C['places']}/{p}" for p in s.path], 'places')
for k in nets: print(k, r[k]['emb'].shape, (r[k]['log'].argmax(1) == s.label.values).mean().round(3))
sb = bd[bd.split == 'val'].sample(480, random_state=0)
rb = extract([f"{C['bdd']}/{p}" for p in sb.path], 'bdd')
for k in nets: print(k, rb[k]['emb'].shape, (rb[k]['log'][:, :11].argmax(1) == sb.cond_id.values).mean().round(3), (rb[k]['log'][:, 11:].argmax(1) == sb.person.values).mean().round(3))

/kaggle/input/datasets/aadiparkhi/drift-multitask-checkpoints


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

resnet18 (480, 512) 0.627
resnet50 (480, 2048) 0.708
convnext (480, 768) 0.765
dino (480, 768) 0.785
resnet18 (480, 512) 0.698 0.84
resnet50 (480, 2048) 0.717 0.915
convnext (480, 768) 0.74 0.892
dino (480, 768) 0.735 0.921


In [3]:
def mk(dom, df, base, lab):
    parts = []
    for name, d in [('val', df[df.split == 'val']), ('test', df[df.split == 'test']),
                    ('train', df[df.split == 'train'].sample(3000, random_state=0))]:
        p = d[['path'] + lab].copy(); p['path'] = C[dom] + '/' + p.path
        p['set'], p['corruption'], p['severity'], p['bid'] = name, 'none', 0, -1
        parts.append(p)
    p = pd.read_csv(f'{G}/corrupt_{dom}_manifest.csv')[['path', 'corruption', 'severity', 'bid']]
    p[lab] = base[lab].values[p.bid.values]
    p['path'] = f'{G}/corrupt/' + p.path
    p['set'] = 'corrupt'
    return pd.concat(parts + [p], ignore_index=True)

bpl, bbd = pd.read_csv(f'{G}/base_places.csv'), pd.read_csv(f'{G}/base_bdd.csv')
meta = {'places': mk('places', pl, bpl, ['label']), 'bdd': mk('bdd', bd, bbd, ['cond_id', 'person'])}

import time
for d, m in meta.items():
    print(d, len(m), m.set.value_counts().to_dict(), 'missing files:', sum(not os.path.exists(p) for p in m.path))
    t = time.time(); r = extract(m[m.set == 'corrupt'].path.sample(256, random_state=0).tolist(), d)
    print({k: v['emb'].shape for k, v in r.items()}, round(time.time() - t, 1), 's for 256 images')

places 36984 {'corrupt': 29184, 'train': 3000, 'test': 2400, 'val': 2400} missing files: 0
{'resnet18': (256, 512), 'resnet50': (256, 2048), 'convnext': (256, 768), 'dino': (256, 768)} 3.1 s for 256 images
bdd 36046 {'corrupt': 30704, 'train': 3000, 'val': 1176, 'test': 1166} missing files: 0
{'resnet18': (256, 512), 'resnet50': (256, 2048), 'convnext': (256, 768), 'dino': (256, 768)} 3.8 s for 256 images


In [4]:
for d, m in meta.items():
    r = extract(m.path.tolist(), d)
    np.savez(f'/kaggle/working/feats_{d}.npz', **{f'{k}_{n}': a for k, dd in r.items() for n, a in dd.items()})
    m.to_csv(f'/kaggle/working/meta_{d}.csv', index=False)
    print(d, 'saved', len(m))
!ls -lh /kaggle/working

places saved 36984
bdd saved 36046
total 614M
-rw-r--r-- 1 root root 289M Oct 10 20:30 feats_bdd.npz
-rw-r--r-- 1 root root 317M Oct 10 20:25 feats_places.npz
-rw-r--r-- 1 root root 4.2M Oct 10 20:30 meta_bdd.csv
-rw-r--r-- 1 root root 4.3M Oct 10 20:25 meta_places.csv
-rw-r--r-- 1 root root  13K Oct 10 20:25 __notebook__.ipynb
